In [18]:
import os
import shutil
import random
from tqdm import tqdm
import torch

import os, random, torch

class Config:
    def __init__(self):
        self.data_root = "/content/processed_data"  # هنا فعلاً الداتا في كولاب

        self.train_dir = os.path.join(self.data_root, "train")
        self.val_dir   = os.path.join(self.data_root, "val")

        self.num_classes   = 7
        self.batch_size    = 64
        self.num_epochs    = 20
        self.learning_rate = 1e-3
        self.weight_decay  = 1e-4
        self.num_workers   = 2
        self.pin_memory    = True
        self.device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.seed          = 42

        self.save_path = "/content/emotion_model_best.pt"
        self.print_every = 1

    def set_seed(self, seed=42):
        random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)


import torch
from torch.utils.data import DataLoader as TorchDataLoader
from torchvision import transforms, datasets

class DataLoader :
    def __init__(self, config: Config, train=True, shuffle=True):
        self.config = config
        self.train = train
        self.shuffle = shuffle
        self.data_root = config.data_root
        self.batch_size = config.batch_size
    
    def get_transforms(self):
        train_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((48, 48)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        
        val_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((48, 48)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        
        return train_transform , val_transform
    
    def create_loaders(self , cfg:Config):
        train_tf , val_tf = self.get_transforms()
        
        train_dataset = datasets.ImageFolder(
            root = cfg.train_dir,
            transform=train_tf
        )
        
        val_dataset = datasets.ImageFolder(
            root= cfg.val_dir,
            transform=val_tf
        )
        
        print(f"train dataset size : {len(train_dataset)}")
        print("train classes" , train_dataset.classes)
        print(f"val dataset size : {len(val_dataset)}")
        print("val classes" , val_dataset.classes)
        
        train_loader = TorchDataLoader(
                train_dataset,
                batch_size=cfg.batch_size,
                shuffle=True,
                num_workers=cfg.num_workers,
                pin_memory=cfg.pin_memory,
                    )

        val_loader = TorchDataLoader(
            val_dataset,
            batch_size=cfg.batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
                )

        print("Train samples:", len(train_dataset))
        print("Val samples:", len(val_dataset))

        return train_loader, val_loader

import os 
import shutil

import torch
from torch import nn
import numpy as np 
import torch.optim as optim



class ConvBlock(nn.Module):
    
    def __init__(self , in_channels :int, out_channels :int):
        super().__init__()
        
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels , out_channels , kernel_size = 3 , padding = 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),
        )
        
        
    def forward(self , x):
        return self.conv_block(x)
    
    


class EmotionCNN (nn.Module):
    def __init__(self , num_classes :int =7):
        super().__init__()
        
        self.features = nn.Sequential(
            ConvBlock(1 , 32),
            ConvBlock(32 , 64),
            ConvBlock(64 , 128),
            ConvBlock(128 , 256),
        )
        
        self.global_pool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256 , num_classes)
        )
        
    def forward(self , x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x




        
        
               

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
# from config import Config
# from model import EmotionCNN , ConvBlock
# from dataLoader import DataLoader
class Trainer:
    def __init__(self, model, train_loader, val_loader, cfg: Config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.cfg = cfg

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=self.cfg.learning_rate,
            weight_decay=self.cfg.weight_decay,
        )

        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.5, patience=3, verbose=True
        )

        self.best_val_acc = 0.0

    def train_one_epoch(self, epoch: int):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in self.train_loader:
            images = images.to(self.cfg.device)
            labels = labels.to(self.cfg.device)

            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        return running_loss / total, correct / total

    def evaluate(self):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in self.val_loader:
                images = images.to(self.cfg.device)
                labels = labels.to(self.cfg.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        return running_loss / total, correct / total

    def fit(self):
        print("Training on:", self.cfg.device)

        for epoch in range(1, self.cfg.num_epochs + 1):
            train_loss, train_acc = self.train_one_epoch(epoch)
            val_loss, val_acc = self.evaluate()

            self.scheduler.step(val_loss)

            if epoch % self.cfg.print_every == 0:
                print(
                    f"Epoch [{epoch}/{self.cfg.num_epochs}] "
                    f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
                    f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
                )

            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                torch.save(self.model.state_dict(), self.cfg.save_path)
                print(
                    f">>> New best model saved! Val Acc = {self.best_val_acc:.4f} "
                    f"-> {self.cfg.save_path}"
                )

        print("Training finished. Best Val Acc:", self.best_val_acc)




# =========================
# 5. Main
# =========================
def main():
    cfg = Config()

    print("train dir:", cfg.train_dir)
    print("val dir  :", cfg.val_dir)

    import os
    for root in [cfg.train_dir, cfg.val_dir]:
        print(f"\nListing {root}:")
        for cls in os.listdir(root):
            cls_path = os.path.join(root, cls)
            if os.path.isdir(cls_path):
                n_files = len([
                    f for f in os.listdir(cls_path)
                    if os.path.isfile(os.path.join(cls_path, f))
                ])
                print(f"  {cls}: {n_files} files")


if __name__ == "__main__":
    main()

train dir: /content/processed_data/train
val dir  : /content/processed_data/val

Listing /content/processed_data/train:


FileNotFoundError: [Errno 2] No such file or directory: '/content/processed_data/train'

In [13]:
import os
import shutil
import random
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader as TorchDataLoader
from torchvision import transforms, datasets
import torch.optim as optim
from pathlib import Path

# ==========================================
# 1. Configuration
# ==========================================
class Config:
    def __init__(self):
        # Auto-detect the correct path based on where script is run from
        self.data_root = self._find_data_root()
        
        self.train_dir = os.path.join(self.data_root, "train")
        self.val_dir   = os.path.join(self.data_root, "val")
        
        self.num_classes   = 7
        self.batch_size    = 64
        self.num_epochs    = 20
        self.learning_rate = 1e-3
        self.weight_decay  = 1e-4
        self.num_workers   = 2
        self.pin_memory    = True
        self.device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.seed          = 42
        
        self.save_path = "emotion_model_best.pt"  # Save locally
        self.print_every = 1
    
    def _find_data_root(self):
        """Find the processed_data directory from various possible locations"""
        possible_paths = [
            # From notebooks folder
            "../services/emotion-detection/Models/Data/processed_data",
            # From project root
            "services/emotion-detection/Models/Data/processed_data",
            # From emotion-detection folder
            "Models/Data/processed_data",
            # From Models folder
            "Data/processed_data",
            # Direct
            "processed_data",
        ]
        
        for path in possible_paths:
            if os.path.exists(path) and os.path.isdir(path):
                print(f"✓ Found data directory: {path}")
                return path
        
        # If not found, print helpful error
        cwd = os.getcwd()
        print(f"\n❌ ERROR: Could not find processed_data directory!")
        print(f"Current working directory: {cwd}")
        print(f"\nSearched in these locations:")
        for path in possible_paths:
            full_path = os.path.abspath(path)
            exists = "✓" if os.path.exists(path) else "✗"
            print(f"  {exists} {full_path}")
        
        print(f"\nPlease ensure you have the data in one of these locations,")
        print(f"or manually set the path using: cfg.data_root = 'your/path/here'")
        
        # Return first path as fallback (will error later with clear message)
        return possible_paths[0]
    
    def set_seed(self, seed=42):
        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)


# ==========================================
# 2. Data Loader
# ==========================================
class DataLoader:
    def __init__(self, config: Config, train=True, shuffle=True):
        self.config = config
        self.train = train
        self.shuffle = shuffle
        self.data_root = config.data_root
        self.batch_size = config.batch_size
    
    def get_transforms(self):
        train_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((48, 48)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        
        val_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((48, 48)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        
        return train_transform, val_transform
    
    def create_loaders(self, cfg: Config):
        train_tf, val_tf = self.get_transforms()
        
        train_dataset = datasets.ImageFolder(
            root=cfg.train_dir,
            transform=train_tf
        )
        
        val_dataset = datasets.ImageFolder(
            root=cfg.val_dir,
            transform=val_tf
        )
        
        print(f"train dataset size: {len(train_dataset)}")
        print("train classes:", train_dataset.classes)
        print(f"val dataset size: {len(val_dataset)}")
        print("val classes:", val_dataset.classes)
        
        train_loader = TorchDataLoader(
            train_dataset,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
        )
        
        val_loader = TorchDataLoader(
            val_dataset,
            batch_size=cfg.batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
        )
        
        return train_loader, val_loader


# ==========================================
# 3. Model Architecture
# ==========================================
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            
            nn.MaxPool2d(2),
        )
    
    def forward(self, x):
        return self.conv_block(x)


class EmotionCNN(nn.Module):
    def __init__(self, num_classes: int = 7):
        super().__init__()
        
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )
        
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


# ==========================================
# 4. Trainer
# ==========================================
class Trainer:
    def __init__(self, model, train_loader, val_loader, cfg: Config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.cfg = cfg
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=self.cfg.learning_rate,
            weight_decay=self.cfg.weight_decay,
        )
        
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.5, patience=3, verbose=True
        )
        
        self.best_val_acc = 0.0
    
    def train_one_epoch(self, epoch: int):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in self.train_loader:
            images = images.to(self.cfg.device)
            labels = labels.to(self.cfg.device)
            
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        return running_loss / total, correct / total
    
    def evaluate(self):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in self.val_loader:
                images = images.to(self.cfg.device)
                labels = labels.to(self.cfg.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        
        return running_loss / total, correct / total
    
    def fit(self):
        print("Training on:", self.cfg.device)
        
        for epoch in range(1, self.cfg.num_epochs + 1):
            train_loss, train_acc = self.train_one_epoch(epoch)
            val_loss, val_acc = self.evaluate()
            
            self.scheduler.step(val_loss)
            
            if epoch % self.cfg.print_every == 0:
                print(
                    f"Epoch [{epoch}/{self.cfg.num_epochs}] "
                    f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
                    f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
                )
            
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                torch.save(self.model.state_dict(), self.cfg.save_path)
                print(
                    f">>> New best model saved! Val Acc = {self.best_val_acc:.4f} "
                    f"-> {self.cfg.save_path}"
                )
        
        print("Training finished. Best Val Acc:", self.best_val_acc)


# ==========================================
# 5. Main
# ==========================================
def main():
    # Initialize config
    cfg = Config()
    cfg.set_seed(cfg.seed)
    
    print("=" * 60)
    print("EMOTION DETECTION CNN TRAINING")
    print("=" * 60)
    print(f"Data root: {cfg.data_root}")
    print(f"Train dir: {cfg.train_dir}")
    print(f"Val dir: {cfg.val_dir}")
    print(f"Device: {cfg.device}")
    print("=" * 60)
    
    # Verify data directories exist
    if not os.path.exists(cfg.train_dir):
        raise FileNotFoundError(
            f"\n❌ Train directory not found: {cfg.train_dir}\n"
            f"Current directory: {os.getcwd()}\n"
            f"Please check that the data exists at this location."
        )
    if not os.path.exists(cfg.val_dir):
        raise FileNotFoundError(
            f"\n❌ Val directory not found: {cfg.val_dir}\n"
            f"Current directory: {os.getcwd()}\n"
            f"Please check that the data exists at this location."
        )
    
    # List dataset contents
    print("\nDataset structure:")
    for root in [cfg.train_dir, cfg.val_dir]:
        print(f"\n{root}:")
        for cls in sorted(os.listdir(root)):
            cls_path = os.path.join(root, cls)
            if os.path.isdir(cls_path):
                n_files = len([
                    f for f in os.listdir(cls_path)
                    if os.path.isfile(os.path.join(cls_path, f))
                ])
                print(f"  {cls}: {n_files} images")
    
    # Create data loaders
    print("\n" + "=" * 60)
    print("Creating data loaders...")
    print("=" * 60)
    data_loader = DataLoader(cfg)
    train_loader, val_loader = data_loader.create_loaders(cfg)
    
    # Initialize model
    print("\n" + "=" * 60)
    print("Initializing model...")
    print("=" * 60)
    model = EmotionCNN(num_classes=cfg.num_classes).to(cfg.device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Train model
    print("\n" + "=" * 60)
    print("Starting training...")
    print("=" * 60)
    trainer = Trainer(model, train_loader, val_loader, cfg)
    trainer.fit()
    
    print("\n" + "=" * 60)
    print("TRAINING COMPLETE!")
    print("=" * 60)


if __name__ == "__main__":
    main()


❌ ERROR: Could not find processed_data directory!
Current working directory: /content

Searched in these locations:
  ✗ /services/emotion-detection/Models/Data/processed_data
  ✗ /content/services/emotion-detection/Models/Data/processed_data
  ✗ /content/Models/Data/processed_data
  ✗ /content/Data/processed_data
  ✗ /content/processed_data

Please ensure you have the data in one of these locations,
or manually set the path using: cfg.data_root = 'your/path/here'
EMOTION DETECTION CNN TRAINING
Data root: ../services/emotion-detection/Models/Data/processed_data
Train dir: ../services/emotion-detection/Models/Data/processed_data/train
Val dir: ../services/emotion-detection/Models/Data/processed_data/val
Device: cpu


FileNotFoundError: 
❌ Train directory not found: ../services/emotion-detection/Models/Data/processed_data/train
Current directory: /content
Please check that the data exists at this location.

In [15]:
import os
import shutil
import random
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader as TorchDataLoader
from torchvision import transforms, datasets
import torch.optim as optim
from pathlib import Path

# ==========================================
# 0. Colab Setup (Run this first in Colab)
# ==========================================
def setup_colab_data():
    """
    Upload and extract your data in Colab.
    
    Instructions:
    1. Zip your processed_data folder locally (with train/ and val/ inside)
    2. Upload the zip to Colab
    3. This function will extract it
    """
    import zipfile
    from google.colab import files
    
    print("=" * 60)
    print("COLAB DATA SETUP")
    print("=" * 60)
    
    # Check if data already exists
    if os.path.exists("/content/processed_data/train"):
        print("✓ Data already exists at /content/processed_data")
        return "/content/processed_data"
    
    print("\nPlease upload your processed_data.zip file")
    print("(This should contain train/ and val/ folders)")
    
    uploaded = files.upload()
    
    if not uploaded:
        raise Exception("No file uploaded!")
    
    zip_name = list(uploaded.keys())[0]
    print(f"\n✓ Uploaded: {zip_name}")
    
    # Extract
    print("Extracting...")
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    
    print("✓ Extraction complete!")
    
    # Find the extracted folder
    possible_paths = [
        "/content/processed_data",
        "/content/Data/processed_data",
        "/content/Models/Data/processed_data",
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            print(f"✓ Found data at: {path}")
            return path
    
    raise Exception("Could not find processed_data folder after extraction")


# ==========================================
# 1. Configuration
# ==========================================
class Config:
    def __init__(self, data_root=None):
        # If data_root provided, use it; otherwise auto-detect
        if data_root:
            self.data_root = data_root
        else:
            self.data_root = self._find_data_root()
        
        self.train_dir = os.path.join(self.data_root, "train")
        self.val_dir   = os.path.join(self.data_root, "val")
        
        self.num_classes   = 7
        self.batch_size    = 64
        self.num_epochs    = 20
        self.learning_rate = 1e-3
        self.weight_decay  = 1e-4
        self.num_workers   = 2 if torch.cuda.is_available() else 0  # 0 workers for CPU
        self.pin_memory    = torch.cuda.is_available()
        self.device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.seed          = 42
        
        self.save_path = "/content/emotion_model_best.pt" if self._is_colab() else "emotion_model_best.pt"
        self.print_every = 1
    
    def _is_colab(self):
        """Check if running in Google Colab"""
        try:
            import google.colab
            return True
        except:
            return False
    
    def _find_data_root(self):
        """Find the processed_data directory from various possible locations"""
        possible_paths = [
            # Colab paths
            "/content/processed_data",
            "/content/Data/processed_data",
            # Local paths from notebooks folder
            "../services/emotion-detection/Models/Data/processed_data",
            # Local paths from project root
            "services/emotion-detection/Models/Data/processed_data",
            # Local paths from emotion-detection folder
            "Models/Data/processed_data",
            # Local paths from Models folder
            "Data/processed_data",
            # Direct
            "processed_data",
        ]
        
        for path in possible_paths:
            train_path = os.path.join(path, "train")
            if os.path.exists(train_path) and os.path.isdir(train_path):
                print(f"✓ Found data directory: {path}")
                return path
        
        # If not found, print helpful error
        cwd = os.getcwd()
        print(f"\n❌ ERROR: Could not find processed_data directory!")
        print(f"Current working directory: {cwd}")
        print(f"\nSearched in these locations:")
        for path in possible_paths:
            full_path = os.path.abspath(path) if not path.startswith("/") else path
            exists = "✓" if os.path.exists(path) else "✗"
            print(f"  {exists} {full_path}")
        
        if self._is_colab():
            print(f"\n💡 TIP: In Colab, run setup_colab_data() first to upload your data")
            print(f"   Or manually upload to /content/processed_data/")
        else:
            print(f"\n💡 TIP: Make sure you're running from the correct directory")
            print(f"   Or set the path manually: cfg.data_root = 'your/path/here'")
        
        # Return first path as fallback (will error later with clear message)
        return possible_paths[0]
    
    def set_seed(self, seed=42):
        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)


# ==========================================
# 2. Data Loader
# ==========================================
class DataLoader:
    def __init__(self, config: Config, train=True, shuffle=True):
        self.config = config
        self.train = train
        self.shuffle = shuffle
        self.data_root = config.data_root
        self.batch_size = config.batch_size
    
    def get_transforms(self):
        train_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((48, 48)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        
        val_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((48, 48)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        
        return train_transform, val_transform
    
    def create_loaders(self, cfg: Config):
        train_tf, val_tf = self.get_transforms()
        
        train_dataset = datasets.ImageFolder(
            root=cfg.train_dir,
            transform=train_tf
        )
        
        val_dataset = datasets.ImageFolder(
            root=cfg.val_dir,
            transform=val_tf
        )
        
        print(f"train dataset size: {len(train_dataset)}")
        print("train classes:", train_dataset.classes)
        print(f"val dataset size: {len(val_dataset)}")
        print("val classes:", val_dataset.classes)
        
        train_loader = TorchDataLoader(
            train_dataset,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
        )
        
        val_loader = TorchDataLoader(
            val_dataset,
            batch_size=cfg.batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
        )
        
        return train_loader, val_loader


# ==========================================
# 3. Model Architecture
# ==========================================
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            
            nn.MaxPool2d(2),
        )
    
    def forward(self, x):
        return self.conv_block(x)


class EmotionCNN(nn.Module):
    def __init__(self, num_classes: int = 7):
        super().__init__()
        
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )
        
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


# ==========================================
# 4. Trainer
# ==========================================
class Trainer:
    def __init__(self, model, train_loader, val_loader, cfg: Config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.cfg = cfg
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=self.cfg.learning_rate,
            weight_decay=self.cfg.weight_decay,
        )
        
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.5, patience=3, verbose=True
        )
        
        self.best_val_acc = 0.0
    
    def train_one_epoch(self, epoch: int):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in self.train_loader:
            images = images.to(self.cfg.device)
            labels = labels.to(self.cfg.device)
            
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        return running_loss / total, correct / total
    
    def evaluate(self):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in self.val_loader:
                images = images.to(self.cfg.device)
                labels = labels.to(self.cfg.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        
        return running_loss / total, correct / total
    
    def fit(self):
        print("Training on:", self.cfg.device)
        
        for epoch in range(1, self.cfg.num_epochs + 1):
            train_loss, train_acc = self.train_one_epoch(epoch)
            val_loss, val_acc = self.evaluate()
            
            self.scheduler.step(val_loss)
            
            if epoch % self.cfg.print_every == 0:
                print(
                    f"Epoch [{epoch}/{self.cfg.num_epochs}] "
                    f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
                    f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
                )
            
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                torch.save(self.model.state_dict(), self.cfg.save_path)
                print(
                    f">>> New best model saved! Val Acc = {self.best_val_acc:.4f} "
                    f"-> {self.cfg.save_path}"
                )
        
        print("Training finished. Best Val Acc:", self.best_val_acc)


# ==========================================
# 5. Main
# ==========================================
def main(data_root=None):
    """
    Main training function
    
    Args:
        data_root: Optional path to processed_data folder. 
                   If None, will auto-detect.
    
    Usage in Colab:
        # First upload data
        data_path = setup_colab_data()  
        # Then train
        main(data_root=data_path)
    """
    # Initialize config
    cfg = Config(data_root=data_root)
    cfg.set_seed(cfg.seed)
    
    print("=" * 60)
    print("EMOTION DETECTION CNN TRAINING")
    print("=" * 60)
    print(f"Data root: {cfg.data_root}")
    print(f"Train dir: {cfg.train_dir}")
    print(f"Val dir: {cfg.val_dir}")
    print(f"Device: {cfg.device}")
    print("=" * 60)
    
    # Verify data directories exist
    if not os.path.exists(cfg.train_dir):
        raise FileNotFoundError(
            f"\n❌ Train directory not found: {cfg.train_dir}\n"
            f"Current directory: {os.getcwd()}\n\n"
            f"In Colab, run: data_path = setup_colab_data(); main(data_root=data_path)\n"
            f"Or manually upload data to /content/processed_data/"
        )
    if not os.path.exists(cfg.val_dir):
        raise FileNotFoundError(
            f"\n❌ Val directory not found: {cfg.val_dir}\n"
            f"Current directory: {os.getcwd()}\n"
            f"Please check that the data exists at this location."
        )
    
    # List dataset contents
    print("\nDataset structure:")
    for root in [cfg.train_dir, cfg.val_dir]:
        print(f"\n{root}:")
        for cls in sorted(os.listdir(root)):
            cls_path = os.path.join(root, cls)
            if os.path.isdir(cls_path):
                n_files = len([
                    f for f in os.listdir(cls_path)
                    if os.path.isfile(os.path.join(cls_path, f))
                ])
                print(f"  {cls}: {n_files} images")
    
    # Create data loaders
    print("\n" + "=" * 60)
    print("Creating data loaders...")
    print("=" * 60)
    data_loader = DataLoader(cfg)
    train_loader, val_loader = data_loader.create_loaders(cfg)
    
    # Initialize model
    print("\n" + "=" * 60)
    print("Initializing model...")
    print("=" * 60)
    model = EmotionCNN(num_classes=cfg.num_classes).to(cfg.device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Train model
    print("\n" + "=" * 60)
    print("Starting training...")
    print("=" * 60)
    trainer = Trainer(model, train_loader, val_loader, cfg)
    trainer.fit()
    
    print("\n" + "=" * 60)
    print("TRAINING COMPLETE!")
    print("=" * 60)
    print(f"Model saved to: {cfg.save_path}")


if __name__ == "__main__":
    # Check if in Colab
    try:
        import google.colab
        print("\n🔔 Running in Google Colab")
        print("=" * 60)
        print("SETUP INSTRUCTIONS:")
        print("=" * 60)
        print("1. First run: data_path = setup_colab_data()")
        print("2. Upload your processed_data.zip")
        print("3. Then run: main(data_root=data_path)")
        print("=" * 60)
    except:
        # Running locally
        main()


🔔 Running in Google Colab
SETUP INSTRUCTIONS:
1. First run: data_path = setup_colab_data()
2. Upload your processed_data.zip
3. Then run: main(data_root=data_path)


In [17]:
import os
import shutil
import random
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader as TorchDataLoader
from torchvision import transforms, datasets
import torch.optim as optim
from pathlib import Path

# ==========================================
# 0. Colab Setup (Run this first in Colab)
# ==========================================
def setup_colab_data():
    """
    Upload and extract your data in Colab.
    
    Instructions:
    1. Zip your processed_data folder locally (with train/ and val/ inside)
    2. Upload the zip to Colab
    3. This function will extract it
    """
    import zipfile
    from google.colab import files
    
    print("=" * 60)
    print("COLAB DATA SETUP")
    print("=" * 60)
    
    # Check if data already exists
    if os.path.exists("/content/processed_data/train"):
        print("✓ Data already exists at /content/processed_data")
        return "/content/processed_data"
    
    print("\nPlease upload your processed_data.zip file")
    print("(This should contain train/ and val/ folders)")
    
    uploaded = files.upload()
    
    if not uploaded:
        raise Exception("No file uploaded!")
    
    zip_name = list(uploaded.keys())[0]
    print(f"\n✓ Uploaded: {zip_name}")
    
    # Extract
    print("Extracting...")
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    
    print("✓ Extraction complete!")
    
    # Find the extracted folder
    possible_paths = [
        "/content/processed_data",
        "/content/Data/processed_data",
        "/content/Models/Data/processed_data",
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            print(f"✓ Found data at: {path}")
            return path
    
    raise Exception("Could not find processed_data folder after extraction")


# ==========================================
# 1. Configuration
# ==========================================
class Config:
    def __init__(self, data_root=None):
        # If data_root provided, use it; otherwise auto-detect
        if data_root:
            self.data_root = data_root
        else:
            self.data_root = self._find_data_root()
        
        self.train_dir = os.path.join(self.data_root, "train")
        self.val_dir   = os.path.join(self.data_root, "val")
        
        self.num_classes   = 7
        self.batch_size    = 64
        self.num_epochs    = 20
        self.learning_rate = 1e-3
        self.weight_decay  = 1e-4
        self.num_workers   = 2 if torch.cuda.is_available() else 0  # 0 workers for CPU
        self.pin_memory    = torch.cuda.is_available()
        self.device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.seed          = 42
        
        self.save_path = "/content/emotion_model_best.pt" if self._is_colab() else "emotion_model_best.pt"
        self.print_every = 1
    
    def _is_colab(self):
        """Check if running in Google Colab"""
        try:
            import google.colab
            return True
        except:
            return False
    
    def _find_data_root(self):
        """Find the processed_data directory from various possible locations"""
        # Get absolute path of current working directory
        cwd = os.getcwd()
        print(f"Current working directory: {cwd}")
        
        possible_paths = [
            # Local paths from notebooks folder
            "../services/emotion-detection/Models/Data/processed_data",
            # Local paths from project root
            "services/emotion-detection/Models/Data/processed_data",
            # Local paths from emotion-detection folder
            "Models/Data/processed_data",
            # Local paths from Models folder
            "Data/processed_data",
            # Direct
            "processed_data",
            # Colab paths (last priority)
            "/content/processed_data",
            "/content/Data/processed_data",
        ]
        
        for path in possible_paths:
            abs_path = os.path.abspath(path)
            train_path = os.path.join(path, "train")
            
            if os.path.exists(train_path) and os.path.isdir(train_path):
                print(f"✓ Found data directory: {path}")
                print(f"  Absolute path: {abs_path}")
                # Verify it has subdirectories
                subdirs = [d for d in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, d))]
                print(f"  Found {len(subdirs)} emotion classes")
                return path
        
        # If not found, print helpful error
        cwd = os.getcwd()
        print(f"\n❌ ERROR: Could not find processed_data directory!")
        print(f"Current working directory: {cwd}")
        print(f"\nSearched in these locations:")
        for path in possible_paths:
            full_path = os.path.abspath(path) if not path.startswith("/") else path
            exists = "✓" if os.path.exists(path) else "✗"
            print(f"  {exists} {full_path}")
        
        if self._is_colab():
            print(f"\n💡 TIP: In Colab, run setup_colab_data() first to upload your data")
            print(f"   Or manually upload to /content/processed_data/")
        else:
            print(f"\n💡 TIP: Make sure you're running from the correct directory")
            print(f"   Or set the path manually: cfg.data_root = 'your/path/here'")
        
        # Return first path as fallback (will error later with clear message)
        return possible_paths[0]
    
    def set_seed(self, seed=42):
        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)


# ==========================================
# 2. Data Loader
# ==========================================
class DataLoader:
    def __init__(self, config: Config, train=True, shuffle=True):
        self.config = config
        self.train = train
        self.shuffle = shuffle
        self.data_root = config.data_root
        self.batch_size = config.batch_size
    
    def get_transforms(self):
        train_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((48, 48)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        
        val_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((48, 48)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        
        return train_transform, val_transform
    
    def create_loaders(self, cfg: Config):
        train_tf, val_tf = self.get_transforms()
        
        train_dataset = datasets.ImageFolder(
            root=cfg.train_dir,
            transform=train_tf
        )
        
        val_dataset = datasets.ImageFolder(
            root=cfg.val_dir,
            transform=val_tf
        )
        
        print(f"train dataset size: {len(train_dataset)}")
        print("train classes:", train_dataset.classes)
        print(f"val dataset size: {len(val_dataset)}")
        print("val classes:", val_dataset.classes)
        
        train_loader = TorchDataLoader(
            train_dataset,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
        )
        
        val_loader = TorchDataLoader(
            val_dataset,
            batch_size=cfg.batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
        )
        
        return train_loader, val_loader


# ==========================================
# 3. Model Architecture
# ==========================================
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            
            nn.MaxPool2d(2),
        )
    
    def forward(self, x):
        return self.conv_block(x)


class EmotionCNN(nn.Module):
    def __init__(self, num_classes: int = 7):
        super().__init__()
        
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )
        
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


# ==========================================
# 4. Trainer
# ==========================================
class Trainer:
    def __init__(self, model, train_loader, val_loader, cfg: Config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.cfg = cfg
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=self.cfg.learning_rate,
            weight_decay=self.cfg.weight_decay,
        )
        
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.5, patience=3, verbose=True
        )
        
        self.best_val_acc = 0.0
    
    def train_one_epoch(self, epoch: int):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in self.train_loader:
            images = images.to(self.cfg.device)
            labels = labels.to(self.cfg.device)
            
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        return running_loss / total, correct / total
    
    def evaluate(self):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in self.val_loader:
                images = images.to(self.cfg.device)
                labels = labels.to(self.cfg.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        
        return running_loss / total, correct / total
    
    def fit(self):
        print("Training on:", self.cfg.device)
        
        for epoch in range(1, self.cfg.num_epochs + 1):
            train_loss, train_acc = self.train_one_epoch(epoch)
            val_loss, val_acc = self.evaluate()
            
            self.scheduler.step(val_loss)
            
            if epoch % self.cfg.print_every == 0:
                print(
                    f"Epoch [{epoch}/{self.cfg.num_epochs}] "
                    f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
                    f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
                )
            
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                torch.save(self.model.state_dict(), self.cfg.save_path)
                print(
                    f">>> New best model saved! Val Acc = {self.best_val_acc:.4f} "
                    f"-> {self.cfg.save_path}"
                )
        
        print("Training finished. Best Val Acc:", self.best_val_acc)


# ==========================================
# 5. Main
# ==========================================
def main(data_root=None):
    """
    Main training function
    
    Args:
        data_root: Optional path to processed_data folder. 
                   If None, will auto-detect.
    
    Usage in Colab:
        # First upload data
        data_path = setup_colab_data()  
        # Then train
        main(data_root=data_path)
    """
    # Initialize config
    cfg = Config(data_root=data_root)
    cfg.set_seed(cfg.seed)
    
    print("=" * 60)
    print("EMOTION DETECTION CNN TRAINING")
    print("=" * 60)
    print(f"Data root: {cfg.data_root}")
    print(f"Train dir: {cfg.train_dir}")
    print(f"Val dir: {cfg.val_dir}")
    print(f"Device: {cfg.device}")
    print("=" * 60)
    
    # Verify data directories exist
    if not os.path.exists(cfg.train_dir):
        raise FileNotFoundError(
            f"\n❌ Train directory not found: {cfg.train_dir}\n"
            f"Current directory: {os.getcwd()}\n\n"
            f"In Colab, run: data_path = setup_colab_data(); main(data_root=data_path)\n"
            f"Or manually upload data to /content/processed_data/"
        )
    if not os.path.exists(cfg.val_dir):
        raise FileNotFoundError(
            f"\n❌ Val directory not found: {cfg.val_dir}\n"
            f"Current directory: {os.getcwd()}\n"
            f"Please check that the data exists at this location."
        )
    
    # List dataset contents
    print("\nDataset structure:")
    for root in [cfg.train_dir, cfg.val_dir]:
        print(f"\n{root}:")
        for cls in sorted(os.listdir(root)):
            cls_path = os.path.join(root, cls)
            if os.path.isdir(cls_path):
                n_files = len([
                    f for f in os.listdir(cls_path)
                    if os.path.isfile(os.path.join(cls_path, f))
                ])
                print(f"  {cls}: {n_files} images")
    
    # Create data loaders
    print("\n" + "=" * 60)
    print("Creating data loaders...")
    print("=" * 60)
    data_loader = DataLoader(cfg)
    train_loader, val_loader = data_loader.create_loaders(cfg)
    
    # Initialize model
    print("\n" + "=" * 60)
    print("Initializing model...")
    print("=" * 60)
    model = EmotionCNN(num_classes=cfg.num_classes).to(cfg.device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Train model
    print("\n" + "=" * 60)
    print("Starting training...")
    print("=" * 60)
    trainer = Trainer(model, train_loader, val_loader, cfg)
    trainer.fit()
    
    print("\n" + "=" * 60)
    print("TRAINING COMPLETE!")
    print("=" * 60)
    print(f"Model saved to: {cfg.save_path}")


if __name__ == "__main__":
    # Check if in Colab
    # try:
    #     import google.colab
    #     print("\n🔔 Running in Google Colab")
    #     print("=" * 60)
    #     print("SETUP INSTRUCTIONS:")
    #     print("=" * 60)
    #     print("1. First run: data_path = setup_colab_data()")
    #     print("2. Upload your processed_data.zip")
    #     print("3. Then run: main(data_root=data_path)")
    #     print("=" * 60)
    # except:
    #     # Running locally
    main()

Current working directory: /content

❌ ERROR: Could not find processed_data directory!
Current working directory: /content

Searched in these locations:
  ✗ /services/emotion-detection/Models/Data/processed_data
  ✗ /content/services/emotion-detection/Models/Data/processed_data
  ✗ /content/Models/Data/processed_data
  ✗ /content/Data/processed_data
  ✗ /content/processed_data
  ✗ /content/processed_data
  ✗ /content/Data/processed_data

💡 TIP: In Colab, run setup_colab_data() first to upload your data
   Or manually upload to /content/processed_data/
EMOTION DETECTION CNN TRAINING
Data root: ../services/emotion-detection/Models/Data/processed_data
Train dir: ../services/emotion-detection/Models/Data/processed_data/train
Val dir: ../services/emotion-detection/Models/Data/processed_data/val
Device: cpu


FileNotFoundError: 
❌ Train directory not found: ../services/emotion-detection/Models/Data/processed_data/train
Current directory: /content

In Colab, run: data_path = setup_colab_data(); main(data_root=data_path)
Or manually upload data to /content/processed_data/